# Lecture 09 — Differentiable Physics: ODE Integrators & Inverse Problems

**PHYG004 / PHY5006, 2026 Spring · Sogang University**
Prof. Young Woo Choi

---

## Where we are

In **Lecture 08 (PINNs)** we used automatic differentiation to take derivatives of a
neural-network *solution* with respect to its inputs $(x, t)$, and folded the PDE
residual into the loss. The autodiff acted on the **solution space**.

Today we keep the same engine — JAX autodiff — but point it somewhere new. We will
build a **differentiable physics simulator** from scratch: a numerical ODE integrator
in which *every step is differentiable*. Then we differentiate a loss not with respect
to the solution, but with respect to the **parameters of the physical model itself**.
That turns a forward simulator into a tool for solving **inverse problems**.

> **One-line contrast with L08.**
> PINN: autodiff on the *solution* → enforce a *known* equation.
> Differentiable physics: autodiff *through the simulator* → recover the *unknown parameters* of the equation.

## What you'll learn today

By the end of this notebook you will be able to:

1. Write Hamilton's equations for a 1D particle and recall why a **symplectic** integrator
   (Störmer–Verlet / leapfrog) conserves energy where naive Euler does not.
2. **Implement a differentiable Störmer–Verlet integrator** in JAX using `jax.lax.scan`,
   so that the *entire trajectory* is one differentiable function of the force-field parameters.
3. Generate a synthetic "experiment": a noisy trajectory in a **1D double-well**
   $V(x) = a\,x^4 - b\,x^2$ with hidden true parameters $(a_0, b_0)$.
4. Build a **trajectory-matching loss** and use `jax.grad` to differentiate it through
   500 integration steps — checking that the gradient vanishes at the true parameters.
5. Run **gradient-based calibration** with Adam to recover $(a_0, b_0)$ from the noisy data,
   to within 5%.
6. Probe the **breaking point**: as observation noise grows, gradient calibration degrades —
   motivating the *likelihood-free* approach (SBI) we will see in **Lecture 19**.

> **Runtime.** Everything here is toy/analytic and runs on the free Colab **CPU** in
> about 1 minute. No GPU, no downloads, no external data.

> **What you should already know.** From earlier lectures: how `jax.grad`, `jax.jit`,
> and `vmap` work (L08); what a potential, kinetic energy, and Hamiltonian are (any
> classical mechanics course). We will *recap* autodiff in three lines, not re-teach it.


## 0. Setup

We only need the core JAX/Optax/matplotlib stack — no physics-specific packages,
because we build the simulator ourselves.


In [ ]:
# Colab: uncomment to install (pre-installed on most Colab runtimes)
# !pip install -q jax jaxlib optax matplotlib

import jax
import jax.numpy as jnp
import jax.random as jr
from jax import grad, jit, vmap
import optax
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

# Reproducibility (repo convention)
KEY = jr.PRNGKey(42)

# Use float64 — symplectic integrators and energy-drift checks deserve it.
jax.config.update("jax_enable_x64", True)

print(f"JAX backend : {jax.default_backend()}")
print(f"JAX version : {jax.__version__}")
print(f"x64 enabled : {jax.config.jax_enable_x64}")


## 1. The physics: a particle in a double-well potential

We study the simplest non-trivial classical system a physicist can love: a single
particle of unit mass moving in one dimension in a **double-well** potential

$$
V(x;\,a,b) \;=\; a\,x^4 \;-\; b\,x^2 , \qquad a,b > 0 .
$$

This is the canonical **bistable** system — it shows up everywhere from Landau theory of
phase transitions to the reaction coordinate of a chemical isomerization. Working in
**reduced (dimensionless) units** ($m=1$, no $\hbar$, no SI), the Hamiltonian is

$$
H(x,p) \;=\; \underbrace{\tfrac{1}{2}\,p^2}_{\text{kinetic } K} \;+\; \underbrace{a\,x^4 - b\,x^2}_{\text{potential } V}.
$$

**Why a double well and not a harmonic oscillator?** A harmonic oscillator has an
*analytic* solution, so an inverse problem there is almost trivial. The double well has
**two symmetric minima** and a **barrier**, giving the student concrete physical quantities
to verify against:

$$
\frac{dV}{dx} = 4a x^3 - 2bx = 0 \;\Rightarrow\;
x^\star = \pm\sqrt{\tfrac{b}{2a}}, \qquad
\Delta V = V(0) - V(x^\star) = \frac{b^2}{4a}\ (\text{barrier height}).
$$

**Our true parameters** will be $(a_0, b_0) = (1.0,\ 2.0)$, giving minima at
$x^\star = \pm 1.0$ and a barrier $\Delta V = 1.0$. Keep these numbers in mind — they are
the "ground truth" our calibration must rediscover.


In [ ]:
# --- The double-well potential and its (analytic) gradient -------------------
def V(x, a, b):
    '''Double-well potential V(x) = a x^4 - b x^2  (reduced units).'''
    return a * x**4 - b * x**2

def dV_dx(x, a, b):
    '''Analytic force-free gradient dV/dx = 4 a x^3 - 2 b x.

    We could equally well get this from jax.grad(V) — and below we *do* let
    autodiff handle derivatives w.r.t. (a,b). Here the closed form is handy.
    '''
    return 4.0 * a * x**3 - 2.0 * b * x

# True (hidden) parameters of the "real" system we will try to recover
A_TRUE, B_TRUE = 1.0, 2.0
x_star = jnp.sqrt(B_TRUE / (2 * A_TRUE))
barrier = B_TRUE**2 / (4 * A_TRUE)
print(f"True parameters (a0, b0) = ({A_TRUE}, {B_TRUE})")
print(f"Minima at x* = +/- {x_star:.3f}")
print(f"Barrier height dV = {barrier:.3f}")


### Visualize the landscape

Let's plot $V(x)$ and mark the two minima and the barrier. This is the energy surface
our particle will roll around on.


In [ ]:
xs = jnp.linspace(-1.8, 1.8, 400)
Vs = V(xs, A_TRUE, B_TRUE)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xs, Vs, lw=2, color="#1f77b4", label=r"$V(x)=a x^4 - b x^2$")
ax.scatter([-x_star, x_star], [V(x_star, A_TRUE, B_TRUE)]*2,
           color="crimson", zorder=5, label="minima")
ax.axhline(V(0.0, A_TRUE, B_TRUE), ls=":", color="gray")
ax.annotate("", xy=(0, 0), xytext=(0, -V(x_star, A_TRUE, B_TRUE)),
            arrowprops=dict(arrowstyle="<->", color="k"))
ax.text(0.08, -barrier/2, r"$\Delta V$", fontsize=12)
ax.set_xlabel("position $x$"); ax.set_ylabel("potential $V(x)$")
ax.set_title("1D double-well (true parameters)")
ax.legend(); fig.tight_layout(); plt.show()


## 2. Integrating the dynamics — and why the integrator matters

Hamilton's equations for $H = \tfrac12 p^2 + V(x)$ are

$$
\dot{x} = \frac{\partial H}{\partial p} = p, \qquad
\dot{p} = -\frac{\partial H}{\partial x} = -V'(x).
$$

We must integrate these numerically. The *naive* choice is **explicit Euler**:

$$
x_{n+1} = x_n + \Delta t\, p_n, \qquad p_{n+1} = p_n - \Delta t\, V'(x_n).
$$

Euler is differentiable and easy — but it **does not conserve energy**. For an
oscillatory system the energy *drifts* (usually grows) without bound, so a long Euler
trajectory is physically wrong no matter how you tune it.

### Störmer–Verlet (leapfrog) — the symplectic fix

The **velocity Verlet** / Störmer–Verlet scheme is *symplectic*: it exactly preserves a
**shadow Hamiltonian** close to $H$, so the true energy only *oscillates* within a tiny
band and never drifts. One step (the "kick–drift–kick" form):

$$
\begin{aligned}
p_{n+1/2} &= p_n - \tfrac{\Delta t}{2}\, V'(x_n) &&\text{(half kick)}\\
x_{n+1}   &= x_n + \Delta t\, p_{n+1/2}          &&\text{(drift)}\\
p_{n+1}   &= p_{n+1/2} - \tfrac{\Delta t}{2}\, V'(x_{n+1}) &&\text{(half kick)}
\end{aligned}
$$

Crucially, **every operation here is differentiable** — additions, multiplications, and
the smooth force $V'$. So the map $(x_n,p_n)\mapsto(x_{n+1},p_{n+1})$ is differentiable,
and composing 500 of them is *still* differentiable. That is the whole point: a
**differentiable simulator**.


## Checkpoint A — implement the differentiable Störmer–Verlet integrator

**Your task.** Implement one Verlet step, then unroll $N$ steps with `jax.lax.scan`.
Using `scan` (instead of a Python `for` loop) keeps the unrolled computation graph
compact and lets JAX differentiate the whole rollout efficiently.

**Verification criteria (physics-based):**
- Energy $E = K + V$ must be conserved: $E_\text{final}/E_\text{initial} \in [0.99, 1.01]$
  (less than 1% drift) — this is the numerical signature of time-reversibility.
- The phase-space trajectory $(x, p)$ must trace **closed-ish loops** (bounded motion),
  not a spiral that flies off to infinity (which is what Euler would give).

The signature `verlet_step(state, dt, grad_V)` is provided. `state = (x, p)`.


In [ ]:
@partial(jit, static_argnames=("n_steps",))
def simulate_verlet(x0, p0, a, b, dt, n_steps):
    '''Differentiable Stoermer-Verlet (velocity Verlet) rollout.

    Returns the full trajectory of positions, momenta, and total energy.
    Every operation is differentiable w.r.t. (x0, p0, a, b), so this whole
    function can be wrapped in jax.grad later.

    Shapes:
        x0, p0 : scalars
        returns xs, ps, Es each of shape (n_steps + 1,)
    '''
    def grad_V(x):
        # Force = -dV/dx ; here we return dV/dx and subtract in the step.
        return 4.0 * a * x**3 - 2.0 * b * x

    def verlet_step(state, _):
        x, p = state
        p_half = p - 0.5 * dt * grad_V(x)      # half kick
        x_new  = x + dt * p_half               # drift
        p_new  = p_half - 0.5 * dt * grad_V(x_new)  # half kick
        new_state = (x_new, p_new)
        return new_state, new_state            # carry, output

    init = (x0, p0)
    _, (xs, ps) = jax.lax.scan(verlet_step, init, xs=None, length=n_steps)

    # prepend the initial condition so trajectory has n_steps+1 points
    xs = jnp.concatenate([jnp.array([x0]), xs])
    ps = jnp.concatenate([jnp.array([p0]), ps])
    Es = 0.5 * ps**2 + V(xs, a, b)             # total energy along the path
    return xs, ps, Es


# --- Run it -----------------------------------------------------------------
DT, N_STEPS = 0.02, 500
X0, P0 = 1.3, 0.0          # start to the right of a minimum, at rest

xs, ps, Es = simulate_verlet(X0, P0, A_TRUE, B_TRUE, DT, N_STEPS)
print(f"trajectory shapes : xs {xs.shape}, ps {ps.shape}, Es {Es.shape}")
print(f"E_initial         : {Es[0]:.6f}")
print(f"E_final           : {Es[-1]:.6f}")
drift = float(Es[-1] / Es[0])
print(f"E_final / E_init  : {drift:.6f}  -> drift {abs(drift-1)*100:.3f}%")
assert 0.99 <= drift <= 1.01, "Energy drift too large — check the Verlet step!"
print("Checkpoint A PASSED: energy conserved to < 1%.")


### Compare against naive Euler (the cautionary tale)

To *see* why we bothered with Verlet, here is the same system integrated with explicit
Euler. Watch the energy climb — the particle artificially gains energy and eventually
escapes the well.


In [ ]:
@partial(jit, static_argnames=("n_steps",))
def simulate_euler(x0, p0, a, b, dt, n_steps):
    '''Explicit (forward) Euler — NOT symplectic. Shown for contrast only.'''
    def step(state, _):
        x, p = state
        x_new = x + dt * p
        p_new = p - dt * (4.0 * a * x**3 - 2.0 * b * x)
        return (x_new, p_new), (x_new, p_new)
    _, (xs, ps) = jax.lax.scan(step, (x0, p0), xs=None, length=n_steps)
    xs = jnp.concatenate([jnp.array([x0]), xs])
    ps = jnp.concatenate([jnp.array([p0]), ps])
    Es = 0.5 * ps**2 + V(xs, a, b)
    return xs, ps, Es

xs_e, ps_e, Es_e = simulate_euler(X0, P0, A_TRUE, B_TRUE, DT, N_STEPS)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
t = jnp.arange(N_STEPS + 1) * DT

# Energy drift comparison
axes[0].plot(t, Es,   label="Verlet (symplectic)", color="#1f77b4")
axes[0].plot(t, Es_e, label="Euler (drifts)",      color="crimson")
axes[0].set_xlabel("time $t$"); axes[0].set_ylabel("total energy $E$")
axes[0].set_title("Energy conservation"); axes[0].legend()

# Phase-space portrait
axes[1].plot(xs,   ps,   lw=1, color="#1f77b4", label="Verlet")
axes[1].plot(xs_e, ps_e, lw=1, color="crimson", alpha=0.7, label="Euler")
axes[1].set_xlabel("position $x$"); axes[1].set_ylabel("momentum $p$")
axes[1].set_title("Phase-space trajectory"); axes[1].legend()
fig.tight_layout(); plt.show()


## 3. The "experiment": generate a noisy observed trajectory

In a real inverse problem, we *observe* data and *don't know* the parameters that
produced it. Here we manufacture exactly that situation:

1. Take the **true, hidden** parameters $(a_0, b_0) = (1.0, 2.0)$.
2. Integrate the dynamics with our (trusted) Verlet simulator to get a clean trajectory $x(t)$.
3. Add **Gaussian observation noise** $\varepsilon \sim \mathcal N(0, \sigma^2)$ with
   $\sigma = 0.05$ to mimic measurement error.

The noisy trajectory `x_obs` is from now on our **only input**. Pretend you have forgotten
$(a_0, b_0)$.


In [ ]:
SIGMA_OBS = 0.05

# (1)-(2) clean ground-truth trajectory from the hidden parameters
xs_true, ps_true, _ = simulate_verlet(X0, P0, A_TRUE, B_TRUE, DT, N_STEPS)

# (3) inject Gaussian observation noise
KEY, sub = jr.split(KEY)
noise = SIGMA_OBS * jr.normal(sub, shape=xs_true.shape)
x_obs = xs_true + noise

print(f"x_obs shape       : {x_obs.shape}")
print(f"observation noise : sigma = {SIGMA_OBS}")
print(f"SNR (rough)       : {float(jnp.std(xs_true)/SIGMA_OBS):.1f}")

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(t, xs_true, color="#1f77b4", lw=1.5, label="true $x(t)$ (hidden)")
ax.scatter(t[::6], x_obs[::6], s=8, color="crimson", alpha=0.6, label="observed (noisy)")
ax.set_xlabel("time $t$"); ax.set_ylabel("position $x$")
ax.set_title("Synthetic experiment: noisy oscillation in the double well")
ax.legend(); fig.tight_layout(); plt.show()


## Checkpoint B — differentiate the trajectory-matching loss

Now the key move. Define a **loss** that measures how well a *candidate* $(a, b)$
reproduces the observed trajectory:

$$
\mathcal L(a,b) \;=\; \frac{1}{N}\sum_{n=0}^{N}\bigl(x^{\text{sim}}_n(a,b) - x^{\text{obs}}_n\bigr)^2 ,
$$

where $x^{\text{sim}}(a,b)$ is produced by **re-running the differentiable simulator** with
those parameters and the *same* initial condition. Because every Verlet step is
differentiable, $\mathcal L$ is a differentiable function of $(a,b)$, and we can ask JAX
for its gradient:

```python
g = jax.grad(traj_loss)(params_ab, x_obs)
```

**Physics analogy.** This loss is exactly an **action/energy** in parameter space: the
minimum sits at the true $(a_0, b_0)$, and $\nabla_{(a,b)}\mathcal L$ is the "force"
pulling our guess toward it.

**Verification criterion.** Evaluated *at* the true parameters $(a_0, b_0)$, the gradient
norm should be **small** (near the bottom of the loss valley) — much smaller than at a
wrong guess.


In [ ]:
def traj_loss(params_ab, x_obs, x0=X0, p0=P0, dt=DT, n_steps=N_STEPS):
    '''MSE between a simulated trajectory at params (a,b) and the observed data.

    params_ab : array([a, b])
    Differentiable w.r.t. params_ab via the differentiable Verlet rollout.
    '''
    a, b = params_ab[0], params_ab[1]
    xs_sim, _, _ = simulate_verlet(x0, p0, a, b, dt, n_steps)
    return jnp.mean((xs_sim - x_obs) ** 2)

# jit the value-and-grad for speed
loss_and_grad = jit(jax.value_and_grad(traj_loss))

# (1) gradient AT the true parameters — should be tiny
params_true = jnp.array([A_TRUE, B_TRUE])
L_true, g_true = loss_and_grad(params_true, x_obs)
print(f"At TRUE  (a,b)=({A_TRUE},{B_TRUE}):  loss={L_true:.5e},  "
      f"grad={np.array(g_true)}, |grad|={jnp.linalg.norm(g_true):.4e}")

# (2) gradient at a WRONG guess — should be much larger
params_wrong = jnp.array([0.4, 0.8])
L_wrong, g_wrong = loss_and_grad(params_wrong, x_obs)
print(f"At WRONG (a,b)=(0.4,0.8):  loss={L_wrong:.5e},  "
      f"grad={np.array(g_wrong)}, |grad|={jnp.linalg.norm(g_wrong):.4e}")

assert jnp.linalg.norm(g_true) < jnp.linalg.norm(g_wrong), \
    "Gradient at the truth should be smaller than at a wrong guess!"
print("\nCheckpoint B PASSED: loss differentiates through 500 Verlet steps; "
      "gradient is near-zero at the true minimum.")


### Look at the loss landscape

Because we have only two parameters, we can *brute-force* the loss surface
$\mathcal L(a,b)$ on a grid and literally see the valley that gradient descent will roll
into. The white star is the true answer.


In [ ]:
# Vectorize the loss over a 2D grid of (a, b) using vmap
a_grid = jnp.linspace(0.4, 1.8, 60)
b_grid = jnp.linspace(1.0, 3.2, 60)
AA, BB = jnp.meshgrid(a_grid, b_grid)
flat = jnp.stack([AA.ravel(), BB.ravel()], axis=1)   # (3600, 2)

loss_grid = vmap(lambda pab: traj_loss(pab, x_obs))(flat).reshape(AA.shape)

fig, ax = plt.subplots(figsize=(6, 5))
cs = ax.contourf(AA, BB, jnp.log10(loss_grid), levels=30, cmap="viridis")
ax.scatter([A_TRUE], [B_TRUE], marker="*", s=260, color="white",
           edgecolor="k", zorder=5, label="true $(a_0,b_0)$")
fig.colorbar(cs, ax=ax, label=r"$\log_{10}\,\mathcal{L}(a,b)$")
ax.set_xlabel("$a$"); ax.set_ylabel("$b$")
ax.set_title("Trajectory-matching loss landscape")
ax.legend(); fig.tight_layout(); plt.show()


## Checkpoint C — gradient-based calibration to recover $(a_0, b_0)$

Time to actually solve the inverse problem. We start from a deliberately **wrong** guess
and run **Adam** for 200 steps, following the gradient of `traj_loss` downhill. This is
*gradient-based calibration*: tuning the parameters of a physical model so its output
matches data, using gradients that flow **through the simulator**.

We start from a wrong-but-reasonable guess $(a,b) = (0.7, 1.4)$ — say, a rough estimate
from a back-of-envelope fit — and let the gradient pull it to the truth.

**Verification criteria:**
- $\dfrac{|a_\text{recovered} - a_0|}{a_0} < 5\%$ **and** $\dfrac{|b_\text{recovered} - b_0|}{b_0} < 5\%$.
- The loss-vs-step curve decreases monotonically (roughly) and flattens.

> **A word of warning (and the seed of L19).** This loss is **non-convex**. Matching a
> *long* oscillatory trajectory means matching its *phase*, and a far-off guess can lock
> onto a spurious local minimum where the simulated and observed oscillations are
> accidentally half-aligned. We will *deliberately trigger* that failure two cells below.
> A good initial guess is part of what makes gradient calibration work — and a key
> limitation it doesn't share with the posterior methods of L19.


In [ ]:
# --- Optimizer setup --------------------------------------------------------
params = jnp.array([0.7, 1.4])          # wrong-but-reasonable starting guess
optimizer = optax.adam(learning_rate=2e-2)
opt_state = optimizer.init(params)

@jit
def update(params, opt_state, x_obs):
    loss, grads = jax.value_and_grad(traj_loss)(params, x_obs)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

N_OPT = 200
history = {"loss": [], "a": [], "b": []}
for step in range(N_OPT):
    params, opt_state, loss = update(params, opt_state, x_obs)
    history["loss"].append(float(loss))
    history["a"].append(float(params[0]))
    history["b"].append(float(params[1]))
    if step % 40 == 0 or step == N_OPT - 1:
        print(f"step {step:3d} | loss {loss:.3e} | "
              f"a={params[0]:.4f} b={params[1]:.4f}")

a_rec, b_rec = float(params[0]), float(params[1])
err_a = abs(a_rec - A_TRUE) / A_TRUE
err_b = abs(b_rec - B_TRUE) / B_TRUE
print(f"\nRecovered (a,b) = ({a_rec:.4f}, {b_rec:.4f})")
print(f"True      (a,b) = ({A_TRUE:.4f}, {B_TRUE:.4f})")
print(f"Relative error  : a {err_a*100:.2f}%, b {err_b*100:.2f}%")
assert err_a < 0.05 and err_b < 0.05, "Calibration did not reach 5% accuracy!"
print("Checkpoint C PASSED: parameters recovered to < 5%.")


### Convergence curves and the recovered potential

Two diagnostics: (left) the loss and the parameter trajectories versus optimization step;
(right) the recovered double-well overlaid on the true one. If calibration worked, the two
potentials should be nearly indistinguishable.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Loss curve
axes[0].semilogy(history["loss"], color="#1f77b4")
axes[0].set_xlabel("Adam step"); axes[0].set_ylabel("loss (log scale)")
axes[0].set_title("Calibration loss")

# Parameter convergence
axes[1].plot(history["a"], label="$a$", color="#1f77b4")
axes[1].plot(history["b"], label="$b$", color="#ff7f0e")
axes[1].axhline(A_TRUE, ls="--", color="#1f77b4", alpha=0.6)
axes[1].axhline(B_TRUE, ls="--", color="#ff7f0e", alpha=0.6)
axes[1].set_xlabel("Adam step"); axes[1].set_ylabel("parameter value")
axes[1].set_title("Parameters -> true values (dashed)"); axes[1].legend()

# True vs recovered potential
axes[2].plot(xs, V(xs, A_TRUE, B_TRUE), lw=2.5, color="k", label="true $V$")
axes[2].plot(xs, V(xs, a_rec, b_rec), lw=1.8, ls="--", color="crimson",
             label="recovered $V$")
axes[2].set_xlabel("$x$"); axes[2].set_ylabel("$V(x)$")
axes[2].set_title("Recovered double-well"); axes[2].legend()
fig.tight_layout(); plt.show()


### The local-minimum trap (run, then worry)

Calibration worked from $(0.7, 1.4)$. Now watch it **fail** from a *farther* guess
$(0.4, 0.8)$ with the *same data and same optimizer*. The gradient is perfectly valid —
it just points into the wrong basin. The optimizer converges confidently to a **wrong**
$(a,b)$ where the simulated oscillation is accidentally phase-matched to the data over
the 500-step window.

This is the Achilles' heel of gradient-based calibration on long trajectories: the loss
is non-convex, so the answer depends on where you start. (Tactics that help in practice:
shorter fitting windows, multi-start, curriculum on trajectory length, or a smoother
observable — but none *guarantees* the global minimum.)


In [ ]:
def calibrate_from(init, x_obs_local, n_opt=200, lr=2e-2):
    '''Adam calibration from a given init; returns recovered (a,b) and final loss.'''
    p = jnp.array(init)
    opt = optax.adam(lr); st = opt.init(p)
    @jit
    def _step(p, st):
        loss, g = jax.value_and_grad(traj_loss)(p, x_obs_local)
        upd, st = opt.update(g, st)
        return optax.apply_updates(p, upd), st, loss
    last = None
    for _ in range(n_opt):
        p, st, last = _step(p, st)
    return float(p[0]), float(p[1]), float(last)

for init in [(0.7, 1.4), (0.4, 0.8)]:
    a_r, b_r, l = calibrate_from(init, x_obs)
    ea, eb = abs(a_r - A_TRUE)/A_TRUE*100, abs(b_r - B_TRUE)/B_TRUE*100
    tag = "converged to truth" if (ea < 5 and eb < 5) else "TRAPPED in local min"
    print(f"init {init} -> (a,b)=({a_r:.3f},{b_r:.3f}) loss={l:.3e} "
          f"err=({ea:.0f}%,{eb:.0f}%)  [{tag}]")


## 4. (Optional extension) When does gradient calibration break?

Gradient-based calibration is powerful **when you have a differentiable forward model and
a decent initial guess**. We already saw one failure mode (local-minimum traps). The other
is **observation noise**: as $\sigma$ grows, a *single* noisy trajectory stops pinning down
$(a,b)$. Let's sweep $\sigma \in \{0.05, 0.2, 0.5, 1.0, 2.0\}$ (well past the $\sigma=0.05$
we used so far) and, because each draw is random, **average over a few noise realizations**
so the trend is meaningful rather than one lucky/unlucky draw.

**Prediction (physical intuition).** With more noise the loss minimum scatters around the
truth, so the recovered $(a,b)$ drifts. Past some noise level a *point estimate* is simply
the wrong object: what you really want is a **distribution** over $(a,b)$ that reports how
uncertain you should be. That distribution — the posterior $p(a,b\mid x_\text{obs})$ — is
exactly what **L19 (SBI)** delivers.


In [ ]:
def calibrate(x_obs_local, n_opt=200, lr=2e-2, init=(0.7, 1.4)):
    '''Run Adam calibration on a given observed trajectory; return recovered (a,b).'''
    p = jnp.array(init)
    opt = optax.adam(lr)
    st = opt.init(p)
    @jit
    def _step(p, st):
        loss, g = jax.value_and_grad(traj_loss)(p, x_obs_local)
        upd, st = opt.update(g, st)
        return optax.apply_updates(p, upd), st, loss
    for _ in range(n_opt):
        p, st, _ = _step(p, st)
    return float(p[0]), float(p[1])

sigmas = [0.05, 0.2, 0.5, 1.0, 2.0]
N_DRAWS = 5                      # average recovery error over this many noise realizations
mean_err, std_err = [], []
key = jr.PRNGKey(0)
for s in sigmas:
    errs = []
    for _ in range(N_DRAWS):
        key, sub = jr.split(key)
        x_noisy = xs_true + s * jr.normal(sub, shape=xs_true.shape)
        a_r, b_r = calibrate(x_noisy)
        # combined relative error (average of |da|/a0 and |db|/b0)
        errs.append(0.5 * (abs(a_r - A_TRUE)/A_TRUE + abs(b_r - B_TRUE)/B_TRUE) * 100)
    errs = np.array(errs)
    mean_err.append(errs.mean()); std_err.append(errs.std())
    print(f"sigma={s:.2f} -> mean recovery error {errs.mean():5.1f}% "
          f"(+/- {errs.std():.1f}% over {N_DRAWS} draws)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(sigmas, mean_err, yerr=std_err, fmt="o-", capsize=4, color="#1f77b4")
ax.axhline(5.0, ls=":", color="gray", label="5% reference")
ax.set_xlabel(r"observation noise $\sigma$")
ax.set_ylabel("mean recovery error (%)")
ax.set_title("Point estimate degrades as noise grows\n(motivates a posterior, not a point)")
ax.legend(); fig.tight_layout(); plt.show()


## 5. Two inverse-problem paradigms — the bridge to L19 (SBI)

Today we solved an inverse problem by **gradient-based calibration**: we required a
*differentiable* simulator and pushed a point estimate $(a,b)$ downhill on a loss. That is
one of the two great families of inverse methods.

| | **Differentiable physics** (today, L09) | **Simulation-Based Inference / SBI** (L19) |
|---|---|---|
| Needs a *differentiable* simulator? | **Yes** — gradients flow through every step | **No** — treats the simulator as a black box |
| What you get out | a **point estimate** $(\hat a, \hat b)$ | a full **posterior** $p(a,b\mid x_\text{obs})$ |
| Handles non-differentiable / stochastic sims? | No | **Yes** |
| Uncertainty quantification | not directly | built in (it's a distribution) |
| Cost | one optimization run | amortized: train once, infer on many observations |
| Breaks when… | data too noisy / loss too rugged | needs many simulator calls to train |

**The same physical system, two ways.** In **Lecture 19** we will return to *this exact
double-well* and infer $(a,b)$ with a *likelihood-free, amortized* neural posterior — no
gradients through the simulator at all — and read off **error bars**, not just a single
number. The noise-sensitivity plot above is the motivation: where gradient calibration
gives up, a posterior over parameters still tells you *how uncertain* you should be.

### Back-pointer to L08, forward-pointer to L19

- **L08 (PINNs):** autodiff on the *solution* of a known equation.
- **L09 (today):** autodiff *through the simulator* to recover the equation's *parameters* — a point estimate.
- **L19 (SBI):** *amortized, likelihood-free* posterior over those same parameters — full uncertainty, no simulator gradients.

Together these three lectures span the modern toolkit for connecting physical models to data.


## Summary

- A **double-well** $V(x)=a x^4 - b x^2$ is a minimal bistable system with two minima at
  $x^\star=\pm\sqrt{b/2a}$ and a barrier $\Delta V = b^2/4a$.
- A **symplectic** (Störmer–Verlet) integrator conserves energy where Euler drifts —
  and, built in JAX with `jax.lax.scan`, it is **fully differentiable**.
- A **differentiable simulator** turns trajectory-matching into a smooth loss in
  *parameter space*; `jax.grad` flows gradients through all 500 steps.
- **Adam calibration** recovered the hidden $(a_0,b_0)$ to within 5% from a single noisy
  trajectory.
- This is **gradient-based** inverse modeling. Its complement — **likelihood-free SBI** —
  arrives in L19, giving full posteriors where gradients fail.

## References

1. N. Thuerey *et al.*, **Physics-Based Deep Learning** (PBDL), TUM, 2021. — differentiable-physics backbone. https://physicsbaseddeeplearning.org
2. E. Hairer, C. Lubich & G. Wanner, **Geometric Numerical Integration** (Springer, 2006). — symplectic integrators and the shadow-Hamiltonian argument.
3. J. Bradbury *et al.*, **JAX: composable transformations of Python+NumPy programs**, 2018. http://github.com/google/jax
4. K. Cranmer, J. Brehmer & G. Louppe, *The frontier of simulation-based inference*, **PNAS** 117, 30055 (2020). — the L19 paradigm.

---

*End of Lecture 09.*
